# 📄 Advanced RAG Chatbot — Monetary Policy Report

This is the upgraded, recruiter-grade version of the basic notebook. Here's exactly what changed and why:

| Basic version | This version | Why it matters |
|---|---|---|
| `PyPDFLoader` (plain text only) | **Docling** (table-aware parsing) | Tables like "Key Macroeconomic Projections" get preserved as real tables instead of scrambled text |
| Fixed-size chunks, no structure | **Section-aware + table-preserving chunking** | A chunk never cuts a table in half, and each chunk knows which chapter/section it came from |
| Dense search only | **Hybrid search** (dense + sparse) | Catches both "meaning" matches and exact-term matches (acronyms like CAB, LSM, WRT, table numbers) |
| Top-k results used as-is | **Cohere Rerank** on top of hybrid search | Re-scores the candidates for relevance before they reach the LLM — the single biggest accuracy lever in RAG |
| No routing | **Query router** (meta vs factual) | "What is this report about?" and "What's the FY27 CAB projection?" get handled differently, on purpose |
| No memory | **Chat memory** | Follow-up questions like "and what about FY26?" work |
| No citations | **Source citations** | Every answer says which section/table it came from |
| No evaluation | **Built-in mini eval** | A handful of test questions with expected keywords, checked automatically — a real recruiter signal |

Everything is still plain functions and simple control flow (`if`, `for`, `while`) — no custom classes to write. The complexity moved into *what* the pipeline does, not how the code is structured.

### Accounts needed (same 3 as before, all free)

| Service | What it's for | Free key |
|---|---|---|
| Cohere | Dense embeddings + reranking | https://dashboard.cohere.com/api-keys |
| Groq | LLM (answer generation + routing) | https://console.groq.com/keys |
| Qdrant Cloud | Hybrid vector database | https://cloud.qdrant.io |

Run every cell top to bottom. Read the markdown above each cell before running it.


## Step 0 — Install everything

This installs Docling (the table-aware PDF parser), the LangChain integrations, and `fastembed` (used for the sparse/keyword side of hybrid search).

⚠️ Docling pulls in some ML libraries and downloads small layout-detection models on first use — this cell can take **3-5 minutes**. That's normal, let it finish.

In [ ]:
!pip install -q docling langchain langchain-community langchain-cohere langchain-groq langchain-qdrant qdrant-client fastembed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 723.1/723.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!pip install -q langchain-text-splitters

## Step 1 — Upload your PDF

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print("Uploaded file:", pdf_filename)

Saving MPR-Aug-2026.pdf to MPR-Aug-2026.pdf
Uploaded file: MPR-Aug-2026.pdf


## Step 2 — Load your API keys from Colab secrets

Instead of typing keys in every time, we read them from Colab's built-in secret manager (the key icon in the left sidebar).

Make sure each secret has **Notebook access** toggled on, and that the names below exactly match what you saved them as.

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["COHERE_API_KEY"] = userdata.get("COHERE_API_KEY")
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

print("All keys loaded from Colab secrets.")

All keys loaded from Colab secrets.


## Step 3 — Parse the PDF with Docling

Unlike `PyPDFLoader`, Docling actually looks at the document's *layout*: it detects headings, paragraphs, and — importantly — tables, and exports the whole thing as clean Markdown. Tables come out as proper Markdown tables (`| col | col |`) instead of jumbled text, and headings come out as `#`, `##`, `###` — which we'll use in the next step to chunk by section.

This step itself can take a minute or two on a document this size.

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert(pdf_filename)
markdown_text = result.document.export_to_markdown()

print(f"Parsed document into {len(markdown_text)} characters of Markdown.")
print("\nPreview:\n")
print(markdown_text[:1000])

[INFO] 2026-08-12 05:23:38,315 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-12 05:23:38,347 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-12 05:23:38,348 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-12 05:23:38,410 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-12 05:23:38,414 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-12 05:23:38,415 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-12 05:23:38,462 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-12 05:23:38,522 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

W0812 05:24:42.662000 535 torch/_inductor/utils.py:1731] [2/0_1] Not enough SMs to use max_autotune_gemm mode


Parsed document into 92463 characters of Markdown.

Preview:

B

<!-- image -->

<!-- image -->

<!-- image -->

## State Bank of Pakistan

## Monetary Policy Report August 2026

## Monetary Policy Committee

Mr. Jameel Ahmad (Chairperson)

Governor, SBP

Mr. Saleem Ullah

Deputy Governor, SBP

Mr. Muhammad Amin Khan Lodhi

Deputy Governor, SBP

Dr. Inayat Hussain

Executive Director, SBP

Mr. Fawad Anwar

Non-Executive Director, SBP Board

Mr. Muhammad Ali Latif

Non-Executive Director, SBP Board

Mr. Najaf Yawar Khan

Non-Executive Director, SBP Board

Dr. Hanid Mukhtar

External Member

Dr. Naved Hamid

External Member

Dr. S. M. Turab Hussain

External Member

## Table of Contents

## Preface

| Executive Summary and Recent Monetary Policy Considerations   | Executive Summary and Recent Monetary Policy Considerations                                               |   1 |
|---------------------------------------------------------------|------------------------------------------------

## Step 4 — Chunk the document (section-aware, table-preserving)

This is the part that actually needed Docling. The plan:

1. First, split the Markdown on headings (`#`, `##`, `###`) — so every chunk knows which chapter/box it belongs to.
2. Within each section, walk through line by line. Any run of consecutive table lines (starting with `|`) gets kept together as **one chunk**, untouched — we never split a table.
3. Any run of normal paragraph text gets split further only if it's too long, using a standard recursive splitter.

This is all done with two small functions and a couple of `for` loops — no classes.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

def split_into_blocks(text):
    """Splits a section's text into (block_text, is_table) pairs.
    Consecutive '|' lines are grouped as one table block.
    Everything else is grouped as normal text blocks."""
    lines = text.split("\n")
    blocks = []
    current_lines = []
    current_is_table = None

    for line in lines:
        line_is_table = line.strip().startswith("|")
        if current_is_table is None:
            current_is_table = line_is_table

        if line_is_table != current_is_table:
            # the type changed -> close off the current block
            blocks.append(("\n".join(current_lines), current_is_table))
            current_lines = []
            current_is_table = line_is_table

        current_lines.append(line)

    if current_lines:
        blocks.append(("\n".join(current_lines), current_is_table))

    return blocks


def chunk_document(markdown_text):
    """Splits the full document into section-aware, table-preserving chunks."""
    headers_to_split_on = [("#", "h1"), ("##", "h2"), ("###", "h3")]
    header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    sections = header_splitter.split_text(markdown_text)

    final_chunks = []

    for section in sections:
        blocks = split_into_blocks(section.page_content)

        for block_text, is_table in blocks:
            block_text = block_text.strip()
            if not block_text:
                continue

            metadata = dict(section.metadata)
            metadata["content_type"] = "table" if is_table else "text"

            if is_table or len(block_text) <= CHUNK_SIZE:
                final_chunks.append(Document(page_content=block_text, metadata=metadata))
            else:
                sub_texts = recursive_splitter.split_text(block_text)
                for sub_text in sub_texts:
                    final_chunks.append(Document(page_content=sub_text, metadata=metadata))

    return final_chunks


chunks = chunk_document(markdown_text)
table_chunks = [c for c in chunks if c.metadata.get("content_type") == "table"]

print(f"Total chunks: {len(chunks)}")
print(f"Table chunks kept intact: {len(table_chunks)}")
print(f"Text chunks: {len(chunks) - len(table_chunks)}")

Total chunks: 84
Table chunks kept intact: 9
Text chunks: 75


In [ ]:
print(table_chunks[3].page_content)
print("\nFrom section:", table_chunks[3].metadata.get("h2") or table_chunks[3].metadata.get("h1"))

| Table 2: Assumptions for Baseline Outlook   | Table 2: Assumptions for Baseline Outlook   | Table 2: Assumptions for Baseline Outlook   |
|---------------------------------------------|---------------------------------------------|---------------------------------------------|
|                                             | MPC                                         | FY27                                        |
| Global Oil Prices 4 ($/bbl.)                | Jul-2026                                    | 80                                          |
|                                             | Jan-2026                                    | 67                                          |
| Global Food Inflation 5                     | Jul-2026                                    | 1.6                                         |
| 6                                           | Jul-2026                                    | 3.1                                         |
| Global GDP Growth 

## Step 5 — Generate a short document summary

For questions like *"what is this document about?"*, retrieval alone isn't reliable — the useful chunks are scattered. Instead, we generate a short summary **once**, up front, using the LLM, and keep it in memory to include in every prompt. Cheap, and solves the "meta question" problem cleanly.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

summary_prompt = f"""Summarize the purpose, scope, and structure of the following document in 3-4 sentences.
This is the beginning of the document:

{markdown_text[:4000]}
"""

doc_summary = llm.invoke(summary_prompt).content
print(doc_summary)

The document is the Monetary Policy Report for August 2026, published by the State Bank of Pakistan (SBP). The report is structured into various chapters, boxes, and figures, providing an in-depth analysis of macroeconomic developments, risks to the outlook, and monetary policy considerations. The scope of the report includes discussions on inflation, global economic trends, and the SBP's policy stance, with contributions from the Monetary Policy Committee members. The report aims to provide a comprehensive overview of the current economic situation and the SBP's monetary policy decisions.


## Step 6 — Set up hybrid embeddings (dense + sparse)

- **Dense embeddings** (Cohere) capture *meaning* — good for paraphrased questions.
- **Sparse embeddings** (FastEmbed, a BM25-style model) capture *exact terms* — good for acronyms and specific numbers like "CAB", "LSM", "Table 2".

Using both together (hybrid search) generally beats either one alone on documents full of domain jargon and figures, like this one.

In [ ]:
from langchain_cohere import CohereEmbeddings
from langchain_qdrant import FastEmbedSparse

dense_embeddings = CohereEmbeddings(model="embed-english-v3.0")
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

print("Dense and sparse embedding models ready.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Dense and sparse embedding models ready.


## Step 7 — Upload chunks to Qdrant Cloud in hybrid mode

This sends every chunk to both embedding models and stores both vectors side by side in your Qdrant Cloud collection, so we can search using both at once.

In [ ]:
from langchain_qdrant import QdrantVectorStore, RetrievalMode

vector_store = QdrantVectorStore.from_documents(
    chunks,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="mpr_report_hybrid",
    retrieval_mode=RetrievalMode.HYBRID,
)

print("All chunks uploaded to Qdrant Cloud in hybrid (dense + sparse) mode.")

All chunks uploaded to Qdrant Cloud in hybrid (dense + sparse) mode.


In [ ]:
QDRANT_API_KEY = test_key

## Step 8 — Add reranking

Hybrid search gives us a decent shortlist, but the ordering isn't always great. Cohere's rerank model looks at the actual question + each candidate chunk together and re-scores them for relevance — much more accurate than similarity search alone. We ask for the top 15 hybrid matches, then rerank down to the best 5.

In [ ]:
import cohere

co = cohere.Client(os.environ["COHERE_API_KEY"])

def hybrid_search_and_rerank(question, k_search=15, k_final=5):
    """Runs hybrid search, then reranks the results, then returns the top k_final chunks."""
    candidates = vector_store.similarity_search(question, k=k_search)

    if not candidates:
        return []

    candidate_texts = [doc.page_content for doc in candidates]

    reranked = co.rerank(
        model="rerank-english-v3.0",
        query=question,
        documents=candidate_texts,
        top_n=min(k_final, len(candidate_texts)),
    )

    top_docs = [candidates[result.index] for result in reranked.results]
    return top_docs

print("hybrid_search_and_rerank() is ready.")

hybrid_search_and_rerank() is ready.


## Step 9 — Query router (meta vs factual)

We ask the LLM a quick, cheap question first: is this question about the document *as a whole* (meta), or asking for a *specific fact or number* (factual)?

- **Meta** questions get answered mainly from the document summary (Step 5) — no need for heavy retrieval.
- **Factual** questions go through the full hybrid search + rerank pipeline (Step 8).

This is a simple `if/else`, not a trained classifier — but it's exactly the kind of design decision that shows you thought about the problem, not just wired up a pipeline.

In [ ]:
def classify_query(question):
    """Returns 'META' or 'FACTUAL'."""
    router_prompt = f"""Classify the question below as exactly one word: META or FACTUAL.

META = the question asks about the document as a whole (its purpose, structure, authors, what it covers).
FACTUAL = the question asks for a specific fact, number, projection, or detail from inside the document.

Question: {question}

Answer with one word only:"""

    result = llm.invoke(router_prompt).content.strip().upper()
    if "META" in result:
        return "META"
    return "FACTUAL"

print("classify_query() is ready.")

classify_query() is ready.


## Step 10 — Chat memory

A simple list holds the last few question/answer pairs. We include the last 3 turns in every new prompt so the chatbot can handle follow-ups like *"and what about FY26?"* No database needed for a single Colab session — just a Python list.

In [ ]:
chat_history = []  # each item: {"question": ..., "answer": ...}

def format_recent_history(max_turns=3):
    recent = chat_history[-max_turns:]
    if not recent:
        return "(no previous conversation)"

    lines = []
    for turn in recent:
        lines.append(f"Q: {turn['question']}\nA: {turn['answer']}")
    return "\n\n".join(lines)

print("Chat memory ready.")

Chat memory ready.


## Step 11 — The full `answer_question` function

This ties everything together:

1. Classify the question (meta vs factual).
2. Retrieve context accordingly — document summary for meta, hybrid search + rerank for factual.
3. Include recent chat history, for follow-ups.
4. Build the final prompt and ask Groq.
5. Return the answer **and** the sources it used, so we can print citations.

In [ ]:
def answer_question(question):
    query_type = classify_query(question)

    if query_type == "META":
        context = f"Document summary:\n{doc_summary}"
        sources = []
    else:
        top_docs = hybrid_search_and_rerank(question)
        context_parts = []
        sources = []
        for doc in top_docs:
            section = doc.metadata.get("h2") or doc.metadata.get("h1") or "Unknown section"
            content_type = doc.metadata.get("content_type", "text")
            context_parts.append(f"[Section: {section} | Type: {content_type}]\n{doc.page_content}")
            sources.append(f"{section} ({content_type})")
        context = "\n\n---\n\n".join(context_parts) if context_parts else "(no relevant chunks found)"

    recent_history = format_recent_history()

    prompt = f"""You are a helpful assistant answering questions about the State Bank of Pakistan's Monetary Policy Report.
Use ONLY the context below to answer. If the answer isn't in the context, say you don't know.

Recent conversation:
{recent_history}

Context:
{context}

Question: {question}

Answer:"""

    answer = llm.invoke(prompt).content

    chat_history.append({"question": question, "answer": answer})

    return answer, sources, query_type

print("answer_question() is ready.")

answer_question() is ready.


## Step 12 — Try it out

Three test questions: one meta, one factual, and one follow-up — to show routing, retrieval, and memory all working.

In [ ]:
answer, sources, qtype = answer_question("What is the purpose of this report?")
print(f"[Routed as: {qtype}]")
print(answer)

[Routed as: META]
The purpose of this report is to provide a comprehensive overview of the current economic situation and the State Bank of Pakistan's monetary policy decisions.


In [ ]:
answer, sources, qtype = answer_question("What is the projected CPI inflation range for FY27?")
print(f"[Routed as: {qtype}]")
print(answer)
print("\nSources used:")
for s in sources:
    print(" -", s)

[Routed as: FACTUAL]
The projected CPI inflation range for FY27 is 5.5 to 7.5 percent.

Sources used:
 - Chapter 1 -Macroeconomic Developments and Outlook (text)
 - Executive Summary and Recent Monetary Policy Considerations (text)
 - Executive Summary and Recent Monetary Policy Considerations (table)
 - Monetary policy considerations since the January 2026 Monetary Policy Committee Meeting (text)
 - Chapter 2 -Risks to the Macroeconomic Outlook (text)


In [ ]:
answer, sources, qtype = answer_question("And what was it in FY26?")
print(f"[Routed as: {qtype}]")
print(answer)

[Routed as: FACTUAL]
The economic growth in FY26 was 3.7 percent.


## Step 13 — Mini evaluation

A tiny, honest way to sanity-check quality: a handful of questions with an expected keyword each. This isn't a full evaluation suite, but it's a real signal — and it's the kind of thing that turns "I built a RAG chatbot" into "I built a RAG chatbot and tested it."

(To go further, look into the `ragas` library for proper faithfulness/relevance scoring.)

In [ ]:
test_cases = [
    {"question": "What is the FY26 real GDP growth rate?", "expected_keyword": "3.7"},
    {"question": "What is SBP's inflation target range?", "expected_keyword": "5"},
    {"question": "Who is the Governor of SBP?", "expected_keyword": "Jameel"},
    {"question": "What caused the surge in global energy prices in 2026?", "expected_keyword": "Middle East"},
]

print(f"{'Question':<55} {'Pass?':<8}")
print("-" * 65)

for case in test_cases:
    answer, _, _ = answer_question(case["question"])
    passed = case["expected_keyword"].lower() in answer.lower()
    status = "PASS" if passed else "CHECK"
    print(f"{case['question']:<55} {status:<8}")

Question                                                Pass?   
-----------------------------------------------------------------
What is the FY26 real GDP growth rate?                  PASS    
What is SBP's inflation target range?                   PASS    
Who is the Governor of SBP?                             PASS    
What caused the surge in global energy prices in 2026?  PASS    


## Step 14 — Chat with the document

Type your question and press Enter. Type `exit` to stop. Every answer is routed, retrieved, reranked, and remembered.

In [ ]:
while True:
    question = input("\nAsk a question (or type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer, sources, qtype = answer_question(question)
    print(f"\n[Routed as: {qtype}]")
    print("Answer:", answer)
    if sources:
        print("\nSources:")
        for s in sources:
            print(" -", s)


Ask a question (or type 'exit' to stop): HI

[Routed as: META]
Answer: Hello. I'm here to help answer your questions about the State Bank of Pakistan's Monetary Policy Report. What would you like to know?

Ask a question (or type 'exit' to stop): WHAT IS THE MOST IMPORTANT THING IN THIS REPORT

[Routed as: META]
Answer: The most important thing in this report is the comprehensive overview of the current economic situation and the State Bank of Pakistan's monetary policy decisions.


KeyboardInterrupt: Interrupted by user

## What to say about this project in an interview

- "I parsed with Docling instead of a plain text extractor because the source document has financial tables that a naive parser would mangle — I verified this by checking table chunk counts after parsing."
- "I used hybrid search because this domain has a lot of acronyms and exact figures that pure dense embeddings sometimes miss."
- "I added a query router because 'what is this document about' and 'what's the FY27 CAB projection' need fundamentally different retrieval strategies, not just bigger k."
- "I evaluated it against a small test set with expected keywords, not just eyeballing outputs."

### Further extensions (optional, one at a time)
- Swap the keyword-based mini-eval for the `ragas` library (faithfulness, answer relevance, context precision).
- Index multiple past Monetary Policy Reports and answer cross-period comparison questions.
- Move chat memory into `RunnableWithMessageHistory` with per-user sessions if you deploy this as a real multi-user app.
- Add a simple Streamlit or Gradio front end so it's not just a notebook.
